# Week 5 — Subqueries and Window Functions: Subqueries
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Write queries that use the result of another query — a subquery nested inside a `SELECT`, `WHERE`, or `FROM` clause
- Understand when a subquery is the natural tool and when a `JOIN` would serve better
- Filter rows with `IN (subquery)`, compare a value against a `(SELECT ...)` scalar, and treat a subquery as a derived table in `FROM`

### Run this first

The setup cell below loads all 8 Olist tables into a SQLite database and connects the
`%%sql` magic to it. It is the same cell as Weeks 1–4 — run it once, wait for
`Database ready.`, and leave it alone.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


## Why this matters

`order_payments` holds 103,886 payment records, each with its own `payment_value`. Finance
wants to know: how many of those payments are "high value" — worth flagging for extra
scrutiny before they clear? There is no fixed threshold for "high value"; it has to mean
*above what a typical payment on this platform looks like*, and "typical" is itself
something SQL has to compute from the same table before it can answer the question.

That is exactly the shape a **subquery** solves: one query (`SELECT AVG(payment_value) FROM
order_payments`) computes a number, and a second query uses that number as a moving target
inside its own `WHERE` clause — all in a single statement, evaluated as one unit rather than
two separate steps you'd have to glue together yourself.

## 1. Scalar subquery in `WHERE` — comparing against a computed value

A **scalar subquery** is a `SELECT` that is guaranteed to return exactly one row and one
column — a single number (or single date, or single string). Because it collapses to one
value, SQLite can drop it anywhere a literal value would be legal: right inside a `WHERE`
comparison, as if you had typed the number yourself.

`WHERE payment_value > (SELECT AVG(payment_value) FROM order_payments)` reads almost like
English: *keep the payment if its value is greater than the average of all payment values*.
The database runs the inner query first, gets back one number, then runs the outer query as
if that number had been hard-coded into the `WHERE` clause. You never have to compute the
average yourself, store it, and paste it in — the subquery does that arithmetic live, every
time the query runs, so it stays correct even as the underlying data changes.

In [ ]:
%%sql
-- Orders with payment value above average
SELECT COUNT(*) AS high_value_orders
FROM order_payments
WHERE payment_value > (SELECT AVG(payment_value) FROM order_payments)   -- Expected: 31,012

Before trusting that count, it's worth seeing the exact number the inner subquery
computed — the threshold every one of those 31,012 payments had to clear.

In [ ]:
%%sql
-- Verify the average the subquery above used as its threshold
SELECT ROUND(AVG(payment_value), 2) AS avg_payment FROM order_payments   -- Expected: R$154.10

## 2. Subquery in `FROM` — using a query's result as a table

A subquery does not have to live inside `WHERE`. Placed inside `FROM`, it becomes a
**derived table**: SQLite runs the inner query first, materializes its result as a
temporary table, and then lets the outer query `SELECT` / `GROUP BY` / aggregate over that
result exactly as if it were a real table in the database.

This matters because some questions genuinely cannot be answered with one `GROUP BY`. "What
is the average number of items per order?" requires two separate aggregation steps: first
count items *per order* (one `GROUP BY order_id`), then average those per-order counts
across all orders (a second aggregation, over the first result). A single `GROUP BY` can only
do one of those at a time — the subquery-in-`FROM` pattern is what lets you chain them.

In [ ]:
%%sql
-- Average items per order (can't do directly with one GROUP BY)
SELECT ROUND(AVG(item_count), 2) AS avg_items_per_order
FROM (
    SELECT order_id, COUNT(*) AS item_count
    FROM order_items
    GROUP BY order_id
)

## 3. `IN` with a subquery — filtering against a set of values

Where a scalar subquery returns one value, an `IN (subquery)` compares a column against an
entire **list** of values that a subquery produces. `orders` doesn't store which state a
customer is in — that lives on `customers` — so to find delivered orders from São Paulo
customers, the inner query first collects every `customer_id` where `customer_state = 'SP'`,
and the outer query keeps only the orders whose `customer_id` shows up in that list.

This is the same *filter using another table's data* job a `JOIN` also does — and for many
questions either tool works. `IN (subquery)` tends to read more naturally when you only need
to filter rows and never need to display or aggregate a column from the second table; reach
for a `JOIN` the moment you need columns from both tables in the output.

> **Data quality note:** the `products` table has two misspelled columns —
> `product_name_lenght` and `product_description_lenght` (both missing the 'g'). This is
> real-world data, not a typo in this course: whichever table you query, you have to use the
> column names as they actually appear, not the "correctly spelled" name you'd expect.

In [ ]:
%%sql
-- Delivered orders from SP customers
SELECT COUNT(*) AS sp_delivered
FROM orders
WHERE order_status = 'delivered'
  AND customer_id IN (
      SELECT customer_id FROM customers WHERE customer_state = 'SP'
  )                                                                     -- Expected: 40,501

---
## 🤖 Using DeepSeek this week

From Week 4 you've been allowed to use DeepSeek to help draft SQL — the same rule applies
here: **draft, run, verify, then trust.** Subqueries are an easy place for an AI-drafted
query to look plausible and be subtly wrong (a `=` where the subquery can return more than
one row, or a subquery filtering the wrong table), so the verify step matters more than
ever.

The protocol:
1. **Ask** DeepSeek: *"Write a SQLite query against `order_payments` that counts rows where
   `payment_value` is above the average `payment_value`, using a subquery."*
2. **Run** whatever it gives you.
3. **Verify** the number against something you already know is correct — you computed this
   exact metric by hand two cells ago, so you have a verified answer to check against before
   you'd ever trust an AI-drafted version on a question you *hadn't* already solved.

In [ ]:
%%sql
-- Step 3 of the protocol: run the DeepSeek-drafted query, then check the number
-- against the value you already verified above before trusting it.
SELECT COUNT(*) AS high_value_orders
FROM order_payments
WHERE payment_value > (SELECT AVG(payment_value) FROM order_payments)   -- Expected: 31,012 (matches Concept 1)

## Going deeper — `NULL` inside a subquery-filtered table

`products.product_category_name` is `NULL` for 610 of the 32,951 products — Olist never
recorded a category for them. `NULL` behaves the same inside subquery filtering as it does
anywhere else in SQL: it is not a value, it is the *absence* of one, so it can never satisfy
`= NULL`. That comparison silently evaluates to "unknown" for every row — never true — and a
`WHERE column = NULL` query returns **zero rows**, with no error to warn you it did the
wrong thing. The only correct way to test for a missing value is `IS NULL` / `IS NOT NULL`.

This matters especially once you start filtering one table using a subquery on another: if
the subquery's result set can contain `NULL` (as `product_category_name` can here), any row
compared against it with `NOT IN` silently returns nothing at all — a trap worth knowing
about even though today's demo queries avoid it.

In [ ]:
%%sql
-- Find products with NULL category — IS NULL is the only comparison that works
SELECT COUNT(*) AS null_category_products
FROM products
WHERE product_category_name IS NULL      -- Expected: 610
-- WRONG (never use): WHERE product_category_name = NULL  -->  always 0 rows, no error

In [ ]:
%%sql
-- Same IS NULL pattern, a different column — always check, never assume a column is complete
SELECT COUNT(*) AS null_weight
FROM products
WHERE product_weight_g IS NULL           -- Expected: 2

## Common mistakes

**Mistake — using `=` against a subquery that can return more than one row.** A scalar
subquery (Concept 1) is safe with `=`, `>`, `<` etc. *only* because `AVG(...)` is guaranteed
to collapse to a single number. The moment your subquery's `SELECT` can return many rows —
like `SELECT customer_id FROM customers WHERE customer_state = 'SP'`, which returns 41,746
rows — comparing a column to it with `=` is invalid: SQLite has no single value to compare
against and raises a "sub-select returns more than one row" style error at run time. The fix
is exactly what Concept 3 already showed: swap `=` for `IN`, which is built to compare
against a whole list.

In [ ]:
%%sql
-- ── COMMON MISTAKE ──────────────────────────────────────────────────
-- WRONG — customer_state = 'SP' matches 41,746 customer_ids, not one, so
-- '=' has no single value to compare against and SQLite errors out:
--   SELECT COUNT(*) FROM orders WHERE order_status = 'delivered'
--     AND customer_id = (SELECT customer_id FROM customers WHERE customer_state = 'SP')
-- CORRECT — use IN, built for comparing against a whole set of values:
SELECT COUNT(*) AS sp_delivered
FROM orders
WHERE order_status = 'delivered'
  AND customer_id IN (
      SELECT customer_id FROM customers WHERE customer_state = 'SP'
  )                                                                     -- Expected: 40,501

## Mini-challenge — your turn

⏱ ~5–10 min

Concept 1 counted the 31,012 payments **above** the average `payment_value`. Now write the
complement: a scalar subquery counting payments **at or below** the average.

Structure it exactly like Concept 1, but flip the comparison operator to `<=`:

```sql
SELECT COUNT(*) AS not_high_value
FROM order_payments
WHERE payment_value <= (SELECT AVG(payment_value) FROM order_payments)
```

**Expected:** 72,874 (103,886 total payments − 31,012 above average).

In [ ]:
%%sql
-- ⏱ ~5-10 min — your turn! Replace the placeholder below with your own query.
SELECT 'write your query here' AS todo

## Session Summary

| Pattern | What it does | Example |
|---|---|---|
| Scalar subquery in `WHERE` | compares a column against one computed value | `WHERE payment_value > (SELECT AVG(payment_value) FROM order_payments)` |
| Subquery in `FROM` | treats a query's result as a temporary table to aggregate again | `FROM (SELECT order_id, COUNT(*) AS item_count FROM order_items GROUP BY order_id)` |
| `IN` with subquery | filters a column against a whole set of values from another table | `WHERE customer_id IN (SELECT customer_id FROM customers WHERE customer_state = 'SP')` |
| `IS NULL` / `IS NOT NULL` | the only valid way to test for a missing value | `WHERE product_category_name IS NULL` |

---
**Coming up Thursday**: **window functions**. You'll rank sellers by revenue with
`RANK() OVER (ORDER BY ...)` — Olist's top seller does R$229,472.63 — build a running total
of monthly orders through 2017 that reveals the November Black Friday spike of 7,544 orders,
and compare `ROW_NUMBER()` against `RANK()` to see how each one breaks ties. Unlike a
`GROUP BY`, a window function computes its answer *without* collapsing the underlying rows. Thursday closes with a **group exercise** that combines both days: use a subquery to find above-average-revenue sellers, then RANK() customer states by average review score.